# 🗺️ Exercise: Okada Fault Modeling using cutde (Bali / Flores Thrust)

Adjust the variables in **Section 2** below (such as `dip`, `slip`, or the `number_of_earthquakes`) and press **Shift + Enter** to recalculate the absolute vertical displacement field map and your profile data misfit metrics dashboard!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cutde.fullspace as FS
import xml.etree.ElementTree as ET
import re
import warnings

print("→ Initializing Pure Local Okada & KML Analysis Workspace...")

# ==============================================================================
# 1. REFERENCE ORIGIN & ROBUST LOCAL KML PARSER (No Internet Needed)
# ==============================================================================
ORIGIN_LON = 114.43
ORIGIN_LAT = -8.15

def parse_kml_line(filepath):
    """Bypasses rigid XML tag schemas to extract coordinates robustly from any local KML structure."""
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        coord_blocks = re.findall(r'<coordinates>(.*?)</coordinates>', content, re.DOTALL)
        x_list, y_list = [], []
        for block in coord_blocks:
            for point in block.strip().split():
                parts = point.split(',')
                if len(parts) >= 2:
                    lon = float(parts[0])
                    lat = float(parts[1])
                    x_km = (lon - ORIGIN_LON) * 111.32 * np.cos(np.deg2rad(ORIGIN_LAT))
                    y_km = (lat - ORIGIN_LAT) * 111.0
                    x_list.append(x_km)
                    y_list.append(y_km)
        if len(x_list) == 0:
            print(f"⚠️ Warning: Found 0 coordinate points inside local file: {filepath}")
        else:
            print(f"✅ Successfully extracted {len(x_list)} points from local file: {filepath}")
        return np.array(x_list), np.array(y_list)
    except Exception as e:
        print(f"❌ Error: Could not read local file {filepath}. Details: {e}")
        return np.array([]), np.array([])

java_x, java_y = parse_kml_line('E-Java-Coastline.kml')
bali_x, bali_y = parse_kml_line('NW-Bali-Coastline.kml')
t3_x, t3_y = parse_kml_line('T3.kml')

# ==============================================================================
# 2. FAULT CONFIGURATION (Students manipulate this section!)
# ==============================================================================
strike = 95.0       # Orientation of the fault plane (degrees clockwise from North)
dip = 25.0          # Shallow thrust dip angle (degrees)
rake = 90.0         # 90 = Pure thrust (positive slip moves hanging wall up-dip)
slip = 5            # Fault displacement PER EARTHQUAKE EVENT (meters)

fault_depth = 4.0   # Depth to top edge of fault plane (km)
fault_length = 25.0 # Total cumulative length (km)
fault_width = 20.0  # Width down-dip (km)
poisson_ratio = 0.25

fault_x_center = 5.0 # Move Left/Right (West/East) along map in km
fault_y_center = 8.0  # Move Down/Up (South/North) along map in km

# --- ADVANCED SEISMOTECTONIC CONTROLS ---
use_tapered_slip = True       # Set to True to activate center-tapered slip physics!
number_of_earthquakes = 50    # Number of repeatable slip cycles to stack cumulative uplift

# ==============================================================================
# 3. DYNAMIC DATA GRID SETUP
# ==============================================================================
grid_xmin = fault_x_center - (fault_length/2) - 25.0
grid_xmax = fault_x_center + (fault_length/2) + 25.0
grid_ymin = fault_y_center - 35.0
grid_ymax = fault_y_center + 35.0

x_grid = np.linspace(grid_xmin, grid_xmax, 80)
y_grid = np.linspace(grid_ymin, grid_ymax, 80)
X, Y = np.meshgrid(x_grid, y_grid)
obs_pts = np.ascontiguousarray(np.vstack([X.flatten(), Y.flatten(), np.zeros_like(X.flatten())]).T)

dip_rad = np.deg2rad(dip)
rake_rad = np.deg2rad(rake)
w_proj = fault_width * np.cos(dip_rad)
d_top = fault_depth
d_bottom = fault_depth + fault_width * np.sin(dip_rad)

# ==============================================================================
# ==============================================================================
# 3B. MULTI-SEGMENT GEOMETRY GENERATOR (Clockwise South-Dipping Winding)
# ==============================================================================
fault_segments_pts = []
fault_segments_slips = []

u_strike = slip * np.cos(rake_rad)
u_dip = slip * np.sin(rake_rad)

if not use_tapered_slip:
    pts = np.array([
        [fault_x_center - fault_length/2, fault_y_center, -d_top],
        [fault_x_center - fault_length/2, fault_y_center - w_proj, -d_bottom],
        [fault_x_center + fault_length/2, fault_y_center - w_proj, -d_bottom],
        [fault_x_center + fault_length/2, fault_y_center, -d_top]
    ])
    fault_segments_pts.append(pts)
    fault_segments_slips.append(np.array([[u_strike, u_dip, 0.0], [u_strike, u_dip, 0.0]]))
else:
    w_len = fault_length * 0.25
    x0 = fault_x_center - fault_length/2
    x1 = x0 + w_len
    x2 = fault_x_center + fault_length/2 - w_len
    x3 = fault_x_center + fault_length/2
    
    # Segment 1: West Wing Taper (from x0 to x1)
    pts1 = np.array([
        [x0, fault_y_center, -d_top],
        [x0, fault_y_center - w_proj, -d_bottom],
        [x1, fault_y_center - w_proj, -d_bottom],
        [x1, fault_y_center, -d_top]
    ])
    fault_segments_pts.append(pts1)
    fault_segments_slips.append(np.array([[u_strike * 0.5, u_dip * 0.5, 0.0], [u_strike * 0.5, u_dip * 0.5, 0.0]]))
    
    # Segment 2: Central Core (from x1 to x2)
    pts2 = np.array([
        [x1, fault_y_center, -d_top],
        [x1, fault_y_center - w_proj, -d_bottom],
        [x2, fault_y_center - w_proj, -d_bottom],
        [x2, fault_y_center, -d_top]
    ])
    fault_segments_pts.append(pts2)
    fault_segments_slips.append(np.array([[u_strike, u_dip, 0.0], [u_strike, u_dip, 0.0]]))
    
    # Segment 3: East Wing Taper (from x2 to x3)
    pts3 = np.array([
        [x2, fault_y_center, -d_top],
        [x2, fault_y_center - w_proj, -d_bottom],
        [x3, fault_y_center - w_proj, -d_bottom],
        [x3, fault_y_center, -d_top]
    ])
    fault_segments_pts.append(pts3)
    fault_segments_slips.append(np.array([[u_strike * 0.5, u_dip * 0.5, 0.0], [u_strike * 0.5, u_dip * 0.5, 0.0]]))
# ==============================================================================
# 3C. EXECUTE SYSTEM SIMULATION LOOP
# ==============================================================================
total_disp = np.zeros_like(obs_pts)
tris = np.array([[0, 1, 2], [0, 2, 3]])

for s_idx in range(len(fault_segments_pts)):
    f_pts = fault_segments_pts[s_idx]
    f_slips = fault_segments_slips[s_idx]
    for idx in range(len(tris)):
        single_tri_array = np.repeat([f_pts[tris[idx]]], obs_pts.shape[0], axis=0)
        single_slip_array = np.repeat([f_slips[idx]], obs_pts.shape[0], axis=0)
        total_disp += FS.disp(obs_pts, single_tri_array, single_slip_array, poisson_ratio)
Z_uplift = (total_disp[:, 2] * number_of_earthquakes).reshape(X.shape)

# ==============================================================================
# 4. COMPUTE UPLIFT DIRECTLY ALONG THE T3 TERRACE PROFILE
# ==============================================================================
if len(t3_x) >= 2:
    t3_dist_axis = np.linspace(0, np.sqrt((t3_x[-1]-t3_x[0])**2 + (t3_y[-1]-t3_y[0])**2), 100)
    profile_pts_x = np.linspace(t3_x[0], t3_x[-1], 100)
    profile_pts_y = np.linspace(t3_y[0], t3_y[-1], 100)
    profile_obs = np.ascontiguousarray(np.vstack([profile_pts_x, profile_pts_y, np.zeros_like(profile_pts_x)]).T)
    t3_disp = np.zeros_like(profile_obs)
    for s_idx in range(len(fault_segments_pts)):
        f_pts = fault_segments_pts[s_idx]
        f_slips = fault_segments_slips[s_idx]
        for idx in range(len(tris)):
            single_tri_array = np.repeat([f_pts[tris[idx]]], profile_obs.shape[0], axis=0)
            single_slip_array = np.repeat([f_slips[idx]], profile_obs.shape[0], axis=0)
            t3_disp += FS.disp(profile_obs, single_tri_array, single_slip_array, poisson_ratio)
    unscaled_model_uplift = t3_disp[:, 2] * number_of_earthquakes
    observed_t3_uplift = np.linspace(100.0, 50.0, 100)
    rmse_misfit = np.sqrt(np.mean((observed_t3_uplift - unscaled_model_uplift)**2))
else:
    rmse_misfit = 0.0

# ==============================================================================
# 5. GRAPHICS LAYOUT GENERATION
# ==============================================================================
fig, (open_ax1, open_ax2) = plt.subplots(1, 2, figsize=(20, 8))
contour = open_ax1.contourf(X, Y, Z_uplift, levels=25, cmap='RdBu_r', alpha=0.85)
fig.colorbar(contour, ax=open_ax1, label=f'Cumulative Uplift over {number_of_earthquakes} Events (meters)')

# Draw the top trace of each fault segment
for f_pts in fault_segments_pts:
    open_ax1.plot([f_pts[0, 0], f_pts[3, 0]], [f_pts[0, 1], f_pts[3, 1]], 'k-', lw=3)
open_ax1.plot([], [], 'k-', lw=3, label='Flores Thrust Segments')

if len(java_x) > 0: open_ax1.plot(java_x, java_y, color='#1e4620', lw=2, label='East Java Coastline')
if len(bali_x) > 0: open_ax1.plot(bali_x, bali_y, color='#0f2066', lw=2, label='NW Bali Coastline')
if len(t3_x) > 0: open_ax1.plot(t3_x, t3_y, color='magenta', lw=4, marker='o', markevery=[0, -1], label='T3 Terrace Track')

plot_xmin = fault_x_center - (fault_length/2) - 10.0
plot_xmax = fault_x_center + (fault_length/2) + 10.0
plot_ymin = fault_y_center - 20.0
plot_ymax = fault_y_center + 10.0
open_ax1.set_xlim(plot_xmin, plot_xmax)
open_ax1.set_ylim(plot_ymin, plot_ymax)
open_ax1.set_title(f"Cumulative Okada Uplift Field Map (Total Events: {number_of_earthquakes})", fontsize=11, fontweight='bold')
open_ax1.set_xlabel('Distance East-West (km)')
open_ax1.set_ylabel('Distance South-North (km)')
open_ax1.axis('equal')
open_ax1.legend(loc='lower left')
open_ax1.grid(True, alpha=0.15)

if len(t3_x) >= 2:
    open_ax2.plot(t3_dist_axis, observed_t3_uplift, 'r-', lw=3, label='Observed T3 Terrace Step (100m -> 50m)')
    open_ax2.plot(t3_dist_axis, unscaled_model_uplift, 'b--', lw=3, label='Unscaled Cumulative Model Profile')
    open_ax2.set_title('Terrace T3 Absolute Elevation Misfit Profile', fontsize=11, fontweight='bold')
    open_ax2.set_xlabel('Distance along Profile from West to East (km)')
    open_ax2.set_ylabel('Cumulative Terrace Elevation (meters)')
    open_ax2.grid(True, alpha=0.3, linestyle=':')
    open_ax2.legend(loc='upper right')
    open_ax2.text(0.05, 0.1, f'Profile Data Misfit (RMSE): {rmse_misfit:.2f} meters', 
             transform=open_ax2.transAxes, fontsize=12, fontweight='bold',
             bbox=dict(facecolor='white', alpha=0.8, boxstyle='round,pad=0.5'))
else:
    open_ax2.text(0.5, 0.5, "KML Data Missing or Invalid", ha='center', va='center')

plt.tight_layout()
plt.show()